In [35]:
from rdkit import Chem
import pandas as pd
from ml_enhance import QuantumFPFileLoader
import numpy as np
from ml_enhance import parallelize

In [36]:
loader = QuantumFPFileLoader("../data/QM9/output")
# loader = QuantumFPFileLoader("../data/QFP_rerun/output")
files = loader.list_output_files()

In [37]:
wanted_properties = ["original_smiles", "output_smiles", "energy", "molecular_dipole_norm", "molecular_polarizability_mean", "homo_lumo_gap", "zero_point_energy", "atomization_energy", "xyz"]

In [38]:
for df in loader.stream_conformer_dataframe(files[0], include_coords=True):
    display(df["original_smiles"])

0    [O:1]([C@@:2]1([C:3]([H:11])([H:12])[H:13])[C:...
Name: original_smiles, dtype: string

In [39]:
def get_min_series(file):
    for df in loader.stream_conformer_dataframe(file, include_coords=True):
        min_series = df.loc[df["energy"].argmin(), wanted_properties]
        return min_series

In [40]:
min_seriess = parallelize(get_min_series, files)

min_df = pd.DataFrame(min_seriess).reset_index()

  0%|          | 4/994 [00:00<00:32, 30.59it/s]

KeyboardInterrupt: 

In [33]:
min_df.head()

,index,original_smiles,output_smiles,energy,molecular_dipole_norm,molecular_polarizability_mean,homo_lumo_gap,zero_point_energy,atomization_energy,xyz
0,0,[O:1]=[C:2]1[N:3]([H:14])[c:4]2[c:5]([H:15])[c...,[O:1]=[C:2]1[N:3]([H:14])[c:4]2[c:5]([H:15])[c...,-34.129004,1.514291,-99.743511,0.089533,92.688240,5.239994,"[[1, -3.215995401162942, -0.6117693337828871, ..."
1,0,[Cl:1][c:2]1[c:3]([H:10])[c:4]([H:11])[c:5]([C...,[Cl:1][c:2]1[c:3]([H:10])[c:4]([H:11])[c:5]([C...,-26.123298,0.965419,-95.249802,0.105970,60.507563,3.330826,"[[1, -2.987291144734709, -0.15605563901242914,..."
2,13,[C:1]([C:2]([C:3]([H:19])([H:20])[H:21])([C:4]...,[C:1]([C:2]([C:3]([H:19])([H:20])[H:21])([C:4]...,-45.124599,0.495893,-97.623141,0.156383,174.625936,7.142925,"[[1, -3.4293012719398734, -1.9108930927712622,..."
3,2,[C:1]([c:2]1[c:3]([H:17])[c:4]([H:18])[c:5]([H...,[C:1]([c:2]1[c:3]([H:17])[c:4]([H:18])[c:5]([H...,-39.730889,0.480588,-80.548992,0.127745,123.883937,5.733924,"[[1, -1.1540116803690164, -2.468296503291047, ..."
4,5,[C:1]([O:2][C:3](=[O:4])[c:5]1[c:6]([H:28])[c:...,[C:1]([O:2][C:3](=[O:4])[c:5]1[c:6]([H:28])[c:...,-72.428696,0.629000,-229.679947,0.064549,141.082325,8.520828,"[[1, -7.082350627805967, 0.9674638834465076, -..."


In [34]:
min_df.to_json("aqsol_mols.json", index=False)

In [65]:
# df = pd.read_json("qm9_mols.json")
df = pd.read_json("aqsol_mols.json")

In [66]:
def test(molecule) -> None:
    smiles = molecule.output_smiles
    mol = Chem.MolFromSmiles(smiles, sanitize=False)

    charge = Chem.GetFormalCharge(mol)

    atom_symbol_map = {atom.GetAtomMapNum(): atom.GetSymbol() for atom in mol.GetAtoms()}
    geometry = molecule.xyz

    print(smiles)
    print(atom_symbol_map)

    print(list(sorted(map(lambda lst: lst[0], geometry))))

    atoms = []
    for atom_idx, x, y, z in geometry:
        sym = atom_symbol_map[int(atom_idx)]
        atoms.append(f"{sym} {x:.8f} {y:.8f} {z:.8f}")

    print("------------------------------------")

In [67]:
for row in df.itertuples():
    if row.original_smiles != row.output_smiles:
        print(row.original_smiles)
        print(row.output_smiles)
        print("\n")

[C:1]([C:2]([C:3]([H:19])([H:20])[H:21])([C:4]([H:22])([H:23])[H:24])[c:5]1[c:6]([H:25])[c:7]([H:26])[c:8]([O:9][C:10]([C:11]2([H:29])[C:12]([H:30])([H:31])[O:13]2)([H:27])[H:28])[c:14]([H:32])[c:15]1[H:33])([H:16])([H:17])[H:18]
[C:1]([C:2]([C:3]([H:19])([H:20])[H:21])([C:4]([H:22])([H:23])[H:24])[c:5]1[c:6]([H:25])[c:7]([H:26])[c:8]([O:9][C:10]([C@:11]2([H:29])[C:12]([H:30])([H:31])[O:13]2)([H:27])[H:28])[c:14]([H:32])[c:15]1[H:33])([H:16])([H:17])[H:18]


[C:1](/[C:2](=[C:3](\[C:4]([H:22])([H:23])[H:24])[C:5](=[O:6])[O:7][C:8]([C:9]([C:10]([C:11]([H:30])([H:31])[H:32])([C:12]([C:13]([C:14](=[C:15]([C:16]([H:38])([H:39])[H:40])[C:17]([H:41])([H:42])[H:43])[H:37])([H:35])[H:36])([H:33])[H:34])[H:29])([H:27])[H:28])([H:25])[H:26])[H:21])([H:18])([H:19])[H:20]
[C:1](/[C:2](=[C:3](\[C:4]([H:22])([H:23])[H:24])[C:5](=[O:6])[O:7][C:8]([C:9]([C@@:10]([C:11]([H:30])([H:31])[H:32])([C:12]([C:13]([C:14](=[C:15]([C:16]([H:38])([H:39])[H:40])[C:17]([H:41])([H:42])[H:43])[H:37])([H:35])[H:36])([H